In [10]:
import pandas as pd

df = pd.read_csv(
    "data/raw/diabetic_data.csv",
    na_values=["?"],
    keep_default_na=False,
    dtype={"payer_code": "string"},
)

print(df.shape)
print(df["patient_nbr"].nunique())
print(df["readmitted"].value_counts(normalize=True))
print(df.isna().mean().sort_values(ascending=False).head(6))

(101766, 50)
71518
readmitted
NO     0.539119
>30    0.349282
<30    0.111599
Name: proportion, dtype: float64
weight               0.968585
medical_specialty    0.490822
payer_code           0.395574
race                 0.022336
diag_3               0.013983
diag_2               0.003518
dtype: float64


In [8]:
print(df["A1Cresult"].value_counts(dropna=False))
print(df["max_glu_serum"].value_counts(dropna=False))

A1Cresult
None    84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64
max_glu_serum
None    96420
Norm     2597
>200     1485
>300     1264
Name: count, dtype: int64


In [9]:
import numpy as np
from pathlib import Path

drop_codes = [11, 13, 14, 19, 20, 21]   # expired / hospice — cannot be readmitted
before = len(df)
df = df[~df["discharge_disposition_id"].isin(drop_codes)].copy()
print(f"dropped {before - len(df)} rows ({before} -> {len(df)})")

df["target"] = (df["readmitted"] == "<30").astype(int)
print("base rate:", df["target"].mean())

patients = df["patient_nbr"].unique()
rng = np.random.default_rng(42)
rng.shuffle(patients)

cut = int(len(patients) * 0.8)
train_patients = set(patients[:cut])

train = df[df["patient_nbr"].isin(train_patients)].copy()
test = df[~df["patient_nbr"].isin(train_patients)].copy()

Path("data/processed").mkdir(parents=True, exist_ok=True)
train.to_csv("data/processed/train.csv", index=False)
test.to_csv("data/processed/test.csv", index=False)

print(len(train), len(test))
print("overlap:", len(set(train["patient_nbr"]) & set(test["patient_nbr"])))

dropped 2423 rows (101766 -> 99343)
base rate: 0.11388824577474005
79578 19765
overlap: 0
